# Semana 15 — Árboles, bagging y random forests

## Contexto
En los slides vimos:
- **Árboles de decisión** y CART (Murphy, Cap. 18.1).
- **Bagging** y bootstrap (Murphy 18.3).
- **Random forests** (Murphy 18.4).
- **Interpretabilidad**: feature importance y partial dependence (Murphy 18.6).

## Objetivo del laboratorio
Llevar las ideas teóricas a código. El notebook tiene **cuatro partes**:
1. Árbol desde cero (Gini, mejor split).
2. Comparación de modelos (Logistic Regression, Decision Tree, Bagging, Random Forest).
3. Hiperparámetros (`max_depth`, `min_samples_leaf`, `n_estimators`, `max_features`).
4. Interpretabilidad (impurity-based importance, permutation importance, PDP).

## Entregable
- Código que corra de extremo a extremo.
- Al menos una gráfica de hiperparámetros vs desempeño.
- Reflexión final (5–8 oraciones) respondiendo las **preguntas guía**.

> Nota: no se encontraron notebooks locales del libro Murphy ni del repo `probml` para estos temas. El dataset es **sintético** y reproducible (sklearn + numpy), inspirado conceptualmente en los ejemplos del Cap. 18.

## 0) Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

RNG = np.random.default_rng(42)
RANDOM_STATE = 42

## Dataset sintético: riesgo de crédito

Construimos un dataset reproducible con variables que se parecen a un problema de riesgo:
- `ingreso`, `deuda`: lognormales.
- `utilizacion`: relacionada con `deuda/ingreso` (correlación a propósito).
- `atrasos_previos`, `consultas_recientes`: conteos (Poisson).
- `antiguedad`: gamma.
- `ruido`: variable irrelevante de control.

Target: `default` ∈ {0, 1}.

> Este dataset **no** es del libro; es didáctico. Lo usamos para conectar ideas con un caso reconocible.

In [ ]:
def make_credit_dataset(n=2000, seed=0):
    rng = np.random.default_rng(seed)
    ingreso = rng.lognormal(mean=10.0, sigma=0.5, size=n)
    deuda = rng.lognormal(mean=9.0, sigma=0.7, size=n)
    util = np.clip(0.4 * (deuda / ingreso) + 0.05 * rng.normal(size=n), 0, 1.5)
    atrasos = rng.poisson(lam=0.6, size=n)
    antig = rng.gamma(shape=2.0, scale=20.0, size=n)
    consultas = rng.poisson(lam=1.5, size=n)
    ruido = rng.normal(size=n)

    z = (
        + 1.6 * (deuda / ingreso)
        + 0.9 * util
        + 0.8 * atrasos
        - 0.02 * antig
        + 0.15 * consultas
        - 4.0
    )
    p = 1.0 / (1.0 + np.exp(-z))
    y = (rng.uniform(size=n) < p).astype(int)

    X = np.column_stack([ingreso, deuda, util, atrasos, antig, consultas, ruido])
    feat = [
        "ingreso", "deuda", "utilizacion", "atrasos_previos",
        "antiguedad", "consultas_recientes", "ruido",
    ]
    return X, y, feat

X, y, FEAT = make_credit_dataset(n=2000, seed=0)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)
print("Train:", Xtr.shape, "Test:", Xte.shape, "Tasa positivos train:", ytr.mean().round(3))

## Parte 1 — Árbol desde cero

Implementamos:
- `gini(y)` — impureza Gini de un grupo.
- `weighted_gini(y_left, y_right)` — promedio ponderado tras un split.
- `best_split(X, y)` — busca la (variable, umbral) que más reduce la impureza.

> Inspirado en la definición de CART (Murphy 18.1). No usa sklearn.

In [ ]:
def gini(y):
    """Gini = 1 - sum_k p_k^2, con p_k = proporcion de cada clase."""
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return float(1.0 - np.sum(p ** 2))

def weighted_gini(y_left, y_right):
    n = len(y_left) + len(y_right)
    if n == 0:
        return 0.0
    return (len(y_left) / n) * gini(y_left) + (len(y_right) / n) * gini(y_right)

def best_split(X, y, n_thresholds=20):
    """Busca el mejor split (j, umbral) que minimiza weighted_gini.

    Para cada feature j, prueba `n_thresholds` cuantiles como candidatos.
    Devuelve un dict con la mejor combinacion encontrada.
    """
    base = gini(y)
    best = {"feature": None, "threshold": None, "impurity": base, "gain": 0.0}
    for j in range(X.shape[1]):
        col = X[:, j]
        qs = np.quantile(col, np.linspace(0.05, 0.95, n_thresholds))
        for t in np.unique(qs):
            mask = col <= t
            yL, yR = y[mask], y[~mask]
            if len(yL) == 0 or len(yR) == 0:
                continue
            imp = weighted_gini(yL, yR)
            if imp < best["impurity"]:
                best = {"feature": j, "threshold": float(t),
                        "impurity": float(imp), "gain": float(base - imp)}
    return best

split = best_split(Xtr, ytr)
print("Mejor split encontrado:")
print(f"  variable : {FEAT[split['feature']]} (col {split['feature']})")
print(f"  umbral   : {split['threshold']:.3f}")
print(f"  Gini hijo: {split['impurity']:.4f}  (Gini padre = {gini(ytr):.4f})")
print(f"  ganancia : {split['gain']:.4f}")

**Observación.** El mejor split de la raíz suele coincidir con la variable más informativa (atrasos, utilización o deuda/ingreso). Esa es la primera "pregunta" que un árbol CART hace en este problema.

## Parte 2 — Comparación de modelos

Comparamos cuatro modelos:
- `LogisticRegression` (con `StandardScaler`).
- `DecisionTreeClassifier` (sin podar).
- `BaggingClassifier` sobre árboles.
- `RandomForestClassifier`.

In [ ]:
models = {
    "LogisticReg": Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=1,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=200, max_features="sqrt",
        random_state=RANDOM_STATE, n_jobs=1,
    ),
}

rows = []
for name, m in models.items():
    m.fit(Xtr, ytr)
    acc_tr = accuracy_score(ytr, m.predict(Xtr))
    acc_te = accuracy_score(yte, m.predict(Xte))
    auc = roc_auc_score(yte, m.predict_proba(Xte)[:, 1])
    rows.append((name, acc_tr, acc_te, auc))
    print(f"{name:13s}  train={acc_tr:.3f}  test={acc_te:.3f}  ROC-AUC={auc:.3f}")

**Lectura típica.**
- `LogisticReg`: train ≈ test (no sobreajusta, pero quizá no captura bien las no linealidades).
- `DecisionTree` sin podar: train muy alto, test bastante más bajo → **overfitting**.
- `Bagging`: estabiliza el árbol; el gap train/test se reduce.
- `RandomForest`: en general el mejor en test.

In [ ]:
rf = models["RandomForest"]
print(classification_report(yte, rf.predict(Xte), digits=3))

## Parte 3 — Hiperparámetros

Variamos `max_depth` (árbol y RF) y `n_estimators` (RF). Reportamos al menos una gráfica.

In [ ]:
depths = list(range(1, 21))
tr_acc, te_acc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE).fit(Xtr, ytr)
    tr_acc.append(t.score(Xtr, ytr))
    te_acc.append(t.score(Xte, yte))

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(depths, tr_acc, marker="o", label="train")
ax.plot(depths, te_acc, marker="s", label="test")
ax.set_xlabel("max_depth")
ax.set_ylabel("accuracy")
ax.set_title("DecisionTree: train vs test conforme aumenta la profundidad")
ax.legend(); ax.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout(); plt.show()

In [ ]:
n_list = [10, 25, 50, 100, 200, 400]
oob_scores, te_scores = [], []
for n in n_list:
    m = RandomForestClassifier(
        n_estimators=n, oob_score=True, bootstrap=True,
        max_features="sqrt", random_state=RANDOM_STATE, n_jobs=1,
    ).fit(Xtr, ytr)
    oob_scores.append(m.oob_score_)
    te_scores.append(m.score(Xte, yte))

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(n_list, oob_scores, marker="o", label="OOB score")
ax.plot(n_list, te_scores, marker="s", label="Test accuracy")
ax.set_xlabel("n_estimators"); ax.set_ylabel("accuracy")
ax.set_title("RandomForest: OOB vs Test conforme crece el número de árboles")
ax.legend(); ax.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout(); plt.show()

**Observa.**
- A partir de cierto número de árboles, agregar más no ayuda.
- El OOB score suele estar muy cerca del test score: es una forma de validación interna útil cuando los datos escasean.

## Parte 4 — Interpretabilidad

Calculamos:
- **Impurity-based importance** (atributo `feature_importances_`).
- **Permutation importance** sobre el set de test.
- Un **partial dependence plot** para una variable.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, max_features="sqrt",
    random_state=RANDOM_STATE, n_jobs=1,
).fit(Xtr, ytr)

imp = rf.feature_importances_
perm = permutation_importance(
    rf, Xte, yte, n_repeats=10, random_state=RANDOM_STATE, n_jobs=1
).importances_mean

order = np.argsort(imp)[::-1]
feat_o = [FEAT[i] for i in order]
x = np.arange(len(FEAT)); w = 0.4
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.bar(x - w/2, imp[order], width=w, label="impurity-based")
ax.bar(x + w/2, perm[order], width=w, label="permutation (test)")
ax.set_xticks(x); ax.set_xticklabels(feat_o, rotation=20, ha="right")
ax.set_ylabel("importancia"); ax.legend(); ax.grid(True, linestyle=":", alpha=0.5)
ax.set_title("Feature importance: dos formas, dos rankings parecidos pero no iguales")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.6))
PartialDependenceDisplay.from_estimator(
    rf, Xtr, features=[FEAT.index("utilizacion")],
    feature_names=FEAT, ax=ax, grid_resolution=40,
)
ax.set_title("PDP: probabilidad media de default vs utilizacion")
plt.tight_layout(); plt.show()

## Reflexión final (entregable)

Responde con claridad y brevedad:

1. ¿Las dos importancias coinciden? Si no, ¿por qué crees?
2. ¿Qué variable parece más importante?
3. ¿Hay riesgo de leakage en este dataset? ¿Y en uno real?
4. ¿Alguna variable podría estar correlacionada con otra y "robar" importancia?
5. ¿Por qué feature importance **no** implica causalidad?

> Recordatorio: random forest es un baseline fuerte, pero no sustituye una buena validación ni una revisión crítica de las features.